## Título do trabalho

In [ ]:
import os
import re
import glob
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, Model, optimizers

# reprodutibilidade
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
rng = np.random.default_rng(SEED)

# device/mixed precision - acelerar o treinamento ao reduzir o consumo de RAM
USE_MIXED_PRECISION = False
if USE_MIXED_PRECISION:
    tf.keras.mixed_precision.set_global_policy("mixed_float16")

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU(s) detected: {[g.name for g in gpus]}")
    except RuntimeError as e:
        print(e)
else:
    print("WARNING: no GPU detected — training will be very slow on CPU.")

print(f"TensorFlow version: {tf.__version__}")

!nvidia-smi #para confirmar que está usando a GPU

GPU(s) detected: ['/physical_device:GPU:0']
TensorFlow version: 2.20.0
Tue May 19 00:13:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        | 

## **Cell 02 - Physical Parameters**
Definição das variáveis do domínio e de inferência, provenientes das configurações da simulação

In [ ]:
# ============================================================
# CELL 2 — PHYSICAL PARAMETERS (hard-coded from Fluent setup)
# ============================================================

# --- Domain geometry ---
X_MIN, X_MAX = 0.0, 280.0     # lateral (perpendicular to flow)
Y_MIN, Y_MAX = 0.0, 700.0     # streamwise (inlet at Y=0, outlet at Y=700)
Z_MIN, Z_MAX = 0.0, 190.0     # vertical

# --- Cylinder geometry ---
D_CYL = 54.0                   # diameter [m]
H_CYL = 65.0                   # height [m]
R_CYL = D_CYL / 2.0

# Cylinder center (confirm from your Fluent setup!)
XC_CYL = 140.0                 # centered laterally in X
YC_CYL = 200.0                 # ~3H from inlet
ZC_BASE = 0.0                  # cylinder sits on ground

# --- Flow conditions ---
V_INF = 17.0                   # inlet velocity magnitude [m/s], direction = +Y
T_INF = 293.15                 # inlet temperature [K]

# --- Thermodynamics and transport (air) ---
P_OP = 101325.0                # operating pressure [Pa]
R_SP = 287.058                 # specific gas constant for air [J/(kg·K)]
MU = 1.7894e-5                 # dynamic viscosity [Pa·s]
CP = 1006.43                   # specific heat [J/(kg·K)]
K_TH = 0.0242                  # thermal conductivity [W/(m·K)]
G_ACC = 9.81                   # gravitational acceleration magnitude [m/s²], direction = -Z
RHO_INF = 1.225

print(f"Domain:    X[{X_MIN},{X_MAX}] Y[{Y_MIN},{Y_MAX}] Z[{Z_MIN},{Z_MAX}]  (m)")
print(f"Cylinder:  D={D_CYL}m H={H_CYL}m centered at ({XC_CYL},{YC_CYL},{ZC_BASE})")
print(f"Flow:      V_inf={V_INF} m/s (+Y)")
print(f"Air:       rho_inf={RHO_INF:.4f} kg/m³, mu={MU:.2e}, k={K_TH}, cp={CP}")

Domain:    X[0.0,280.0] Y[0.0,700.0] Z[0.0,190.0]  (m)
Cylinder:  D=54.0m H=65.0m centered at (140.0,200.0,0.0)
Flow:      V_inf=17.0 m/s (+Y)
Air:       rho_inf=1.2250 kg/m³, mu=1.79e-05, k=0.0242, cp=1006.43


## **Cell 03 - Discover and Parse Snapshot Filenames**

Descobrir como que a simulação está guardada (no nosso caso, pelo drive - nomes e formas de separar os timestamps) e dividir os arquivos.

In [ ]:
# ============================================================
# CELL 3 — DISCOVER AND PARSE SNAPSHOT FILENAMES
# ============================================================
# Generic snapshot discovery: works with any directory and any filename
# pattern, as long as the timestamp is the LAST number in the basename.
#
# Subsampling: SNAPSHOT_STRIDE controls temporal subsampling.
# Example: 600 files × dt=0.1s = 60s of physical time.
#   stride=1  → uses all 600 frames (full resolution)
#   stride=20 → uses 30 frames spaced 2.0s apart (good for PINN training)
#   stride=10 → uses 60 frames spaced 1.0s apart (intermediate)

# ----- USER CONFIG -----
SNAPSHOT_DIR      = "/content/drive/MyDrive/IMT/mauá/tcc/CFD e Dados/CSVs CFD Final Sem1 - Sem Energia e Temp/" #caminho para a conta da yasmin
SNAPSHOT_EXT      = ".csv"
DT_PHYSICAL       = 0.1          # tempo físico por timestep index do Fluent
TIMESTEP_IS_INDEX = True

# Stride temporal: pega 1 arquivo a cada SNAPSHOT_STRIDE
SNAPSHOT_STRIDE   = 1           # 20 → 30 frames (de 600), espaçados 2.0s
SNAPSHOT_START    = 0            # índice do primeiro arquivo a usar (default: 0)
SNAPSHOT_MAX      = None         # int para limitar total; None = sem limite

# Pre-compiled pattern: o último número não-sinalizado no basename
_TIME_TOKEN_RE = re.compile(r"(\d+(?:\.\d+)?)(?=\D*$)")


def parse_timestamp(filename: str,
                    dt_physical: float = DT_PHYSICAL,
                    timestep_is_index: bool = TIMESTEP_IS_INDEX):
    """
    Extract physical time (in seconds) from a snapshot filename.

    Returns
    -------
    t_physical : float | None
    raw_value  : float | None
    """
    stem = os.path.splitext(os.path.basename(filename))[0]
    matches = _TIME_TOKEN_RE.findall(stem)
    if not matches:
        return None, None
    raw = float(matches[-1])
    t_phys = raw * dt_physical if timestep_is_index else raw
    return float(t_phys), raw


def discover_snapshots(snapshot_dir: str,
                       ext: str = SNAPSHOT_EXT,
                       dt_physical: float = DT_PHYSICAL,
                       timestep_is_index: bool = TIMESTEP_IS_INDEX,
                       stride: int = 1,
                       start_idx: int = 0,
                       max_snapshots: int = None,
                       verbose: bool = True):
    """
    Scan `snapshot_dir` for snapshot files and return a stride-subsampled list.

    Parameters
    ----------
    stride : int
        Take every Nth file (after sorting by physical time).
        stride=1 means no subsampling.
    start_idx : int
        Skip the first `start_idx` files (useful to drop initial transient).
    max_snapshots : int or None
        Hard cap on number of returned snapshots (after stride applied).

    Returns
    -------
    list of (t_physical, filepath) tuples in chronological order.
    """
    if not os.path.isdir(snapshot_dir):
        raise NotADirectoryError(f"Snapshot directory not found: {snapshot_dir}")

    pattern = os.path.join(snapshot_dir, f"*{ext}")
    candidates = sorted(glob.glob(pattern))
    if verbose:
        print(f"Scanning '{snapshot_dir}' for *{ext}: "
              f"{len(candidates)} candidate(s)")

    # Step 1: parse timestamps and filter unparseable
    parsed = []
    skipped = []
    for path in candidates:
        t_phys, raw = parse_timestamp(path, dt_physical, timestep_is_index)
        if t_phys is None:
            skipped.append(os.path.basename(path))
            continue
        parsed.append((t_phys, path))

    # Step 2: sort chronologically
    parsed.sort(key=lambda s: s[0])

    # Step 3: apply stride subsampling
    n_total = len(parsed)
    selected = parsed[start_idx::stride]
    if max_snapshots is not None:
        selected = selected[:max_snapshots]

    if verbose:
        if skipped:
            print(f"  [skip] {len(skipped)} unparseable file(s), "
                  f"e.g. {skipped[:3]}")
        print(f"  [filter] {n_total} parseable → {len(selected)} after "
              f"stride={stride}, start={start_idx}, max={max_snapshots}")

    return selected


# ----- run discovery -----
snaps = discover_snapshots(
    SNAPSHOT_DIR,
    stride=SNAPSHOT_STRIDE,
    start_idx=SNAPSHOT_START,
    max_snapshots=SNAPSHOT_MAX,
)

if len(snaps) == 0:
    raise RuntimeError(
        f"No usable '*{SNAPSHOT_EXT}' files in '{SNAPSHOT_DIR}' "
        f"(stride={SNAPSHOT_STRIDE}, start={SNAPSHOT_START})."
    )

times_all = np.array([t for t, _ in snaps], dtype=np.float32)
print(f"\nFinal snapshot selection: {len(snaps)} frames")
print(f"  Time range:    {times_all[0]:.4f} s → {times_all[-1]:.4f} s")
if len(times_all) > 1:
    dts = np.diff(times_all)
    print(f"  dt (between selected snaps): "
          f"mean={dts.mean():.4f}s  min={dts.min():.4f}s  max={dts.max():.4f}s")
    expected_dt = DT_PHYSICAL * SNAPSHOT_STRIDE
    print(f"  Expected dt (DT_PHYSICAL × stride): {expected_dt:.4f}s")
print(f"  First 3 selected files: {[os.path.basename(p) for _, p in snaps[:3]]}")
print(f"  Last  3 selected files: {[os.path.basename(p) for _, p in snaps[-3:]]}")

NotADirectoryError: Snapshot directory not found: /content/drive/MyDrive/IMT/mauá/tcc/CFD e Dados/CSVs CFD Final Sem1 - Sem Energia e Temp/

## **Cell 04 - Normalization Constants**
Definição das variáveis que vão normalizar os valores

### variáveis físicas:
- variável de dimensão: maior valor entre x, y e z
- variável de velocidade: velocidade dada (17m/s)
- variável de pressão = pressão dinâmica * densidade de inferência * velocidade dada^2
- variável de tempo = dimensão normalizada / velocidade normalizada
  - variável de tempo inicial = tempo inicial / tempo normalizado
  - variável de tempo final = tempo final / tempo normalizado

### variáveis do cilindro:
- centro do cilindro (para x, y) = valor dado naquele eixo do cilindro / dimensão normalizada
- raio do cilindro = raio do cilindro dado / dimensão normalizada
- altura do cilindro = altura do cilindro dada / dimensão normalizada

### condições de contorno:
- velocidade de inlet.

In [ ]:
# ============================================================
# CELL 4 — NORMALIZATION CONSTANTS
# ============================================================
# These scales are derived purely from the physical setup (Cell 2) and the
# discovered time range (Cell 3). They are computed BEFORE loading the bulk
# data so the loader (Cell 5) can normalize each snapshot on the fly,
# avoiding a second pass over RAM.

L_REF   = max(X_MAX - X_MIN, Y_MAX - Y_MIN, Z_MAX - Z_MIN)   # [m]
V_REF   = V_INF                                              # [m/s]
P_REF   = 0.5 * RHO_INF * V_INF**2                     # dynamic pressure scale [Pa]
TIME_REF = L_REF / V_REF                                     # [s]

# Time domain (normalized)
TN_MIN = float(times_all[0])  / TIME_REF
TN_MAX = float(times_all[-1]) / TIME_REF

# Normalized geometry
XC_N = XC_CYL / L_REF
YC_N = YC_CYL / L_REF
R_N  = R_CYL  / L_REF
H_N  = H_CYL  / L_REF
XN_MIN, XN_MAX = X_MIN / L_REF, X_MAX / L_REF
YN_MIN, YN_MAX = Y_MIN / L_REF, Y_MAX / L_REF
ZN_MIN, ZN_MAX = Z_MIN / L_REF, Z_MAX / L_REF

# Target normalized BC values (reused later)
VN_INLET   = V_INF / V_REF                # = 1.0

print(f"Normalization scales:")
print(f"  L_ref  = {L_REF:.2f} m")
print(f"  V_ref  = {V_REF:.2f} m/s")
print(f"  P_ref  = {P_REF:.3f} Pa")
print(f"  t_ref  = {TIME_REF:.4f} s")
print(f"Normalized domain: "
      f"X[{XN_MIN:.3f},{XN_MAX:.3f}] "
      f"Y[{YN_MIN:.3f},{YN_MAX:.3f}] "
      f"Z[{ZN_MIN:.3f},{ZN_MAX:.3f}] "
      f"t[{TN_MIN:.4f},{TN_MAX:.4f}]")

## **Cell 05 - Load and Subsample Snapshots**
ler cada um dos snapshots, aplicar a máscara (tirar os pontos do cilindro), pegar uma amostra dos pontos (e normalizá-la) e copiar para um array para otimização de memória.
- nome da variável: `data_np`
- pontos por snapshot: 10.000

<font color="pink"> formato final do tensor para cada snapshot: (10.000, 8) = [xn, yn, zn, tn, un, vn, wn, Pn]
- onde:
  - x norm,
  - y norm,
  - z norm,
  - tempo norm,
  - vel em x norm,
  - vel em y norm,
  - vel em z norm,
  - pressão norm.
  </font>

In [ ]:
# ============================================================
# CELL 5 — LOAD AND SUBSAMPLE SNAPSHOTS (memory-efficient)
# ============================================================
# Strategy:
#   * Each snapshot is read, masked (cylinder interior excluded), subsampled,
#     normalized, and IMMEDIATELY copied into a pre-allocated float32 array.
#   * The per-snapshot DataFrame goes out of scope right after -> the GC can
#     reclaim it before the next file is opened. This is the key trick to
#     avoid Colab RAM spikes when you have hundreds of large CFD dumps.
#   * Final tensor shape: (N_total, 8) = [xn, yn, zn, tn, un, vn, wn, Pn]

# ---------- USER CONFIG ----------
N_POINTS_PER_SNAP = 10000      # quantos pontos estamos pegando de cada arquivo - isso é o que ele estava questionando hoje no sábado
MAX_SNAPSHOTS     = None      # None = use all; int = cap (for fast tests)
engine = "c"
# ----------------------------------

# Column-name resolution: list candidates in priority order. Case-insensitive.
NEEDED_COLS = {
    "x": ["x-coordinate"],
    "y": ["y-coordinate"],
    "z": ["z-coordinate"],
    "P": ["pressure"],
    "u": ["x-velocity", "u"],
    "v": ["y-velocity", "v"],
    "w": ["z-velocity", "w"],
}


def resolve_column_names(df_cols):
    """Map our canonical keys (x,y,z,P,u,v,w) to actual DataFrame columns."""
    norm = {c.strip().lower(): c for c in df_cols}
    mapping = {}
    for key, candidates in NEEDED_COLS.items():
        for cand in candidates:
            if cand in norm:
                mapping[key] = norm[cand]
                break
        else:
            raise KeyError(
                f"Snapshot is missing a column for '{key}'. "
                f"Tried: {candidates}. Available: {list(df_cols)}"
            )
    return mapping

# Pre-compute lowercase set of all candidate column names (for usecols filter).
# pandas automatically keeps only the FIRST occurrence when duplicate column
# names appear in the CSV (which Fluent dumps DO have for x/y/z), so we just
# need a stateless filter.
_ALL_CANDIDATES_LOWER = {c for cands in NEEDED_COLS.values() for c in cands}

def _usecols_filter(colname):
    return colname.strip().lower() in _ALL_CANDIDATES_LOWER


def load_snapshot(path, col_map=None):
    """
    Read a single snapshot file into a dict of float32 numpy arrays.
    Uses pandas C engine + usecols + dtype=float32 for ~13x speedup.
    """
    df = pd.read_csv(
        path,
        skipinitialspace=True,                # tolera espaços depois das vírgulas
        usecols=_usecols_filter,              # só as 8 colunas que precisamos
        dtype=np.float32,                     # parse direto em float32
        engine="c",                           # ~13x mais rápido que python
    )
    df.columns = [c.strip() for c in df.columns]

    if col_map is None:
        col_map = resolve_column_names(df.columns)

    out = {k: df[col_map[k]].to_numpy(dtype=np.float32, copy=False)
            for k in NEEDED_COLS}
    return out, col_map


def _subsample_indices(n_valid, n_take, rng):
    """Without-replacement sample if possible; else with-replacement."""
    if n_valid == 0:
        return np.empty(0, dtype=np.int64)
    if n_take >= n_valid:
        return np.arange(n_valid)
    return rng.choice(n_valid, size=n_take, replace=False)


def load_all_snapshots(snaps,
                       n_points_per_snap=N_POINTS_PER_SNAP,
                       max_snapshots=MAX_SNAPSHOTS,
                       seed=SEED,
                       keep_first_raw=True,
                       verbose_every=20):
    """
    Stream-load all snapshots into a single pre-allocated float32 tensor.

    Returns
    -------
    data_np : np.ndarray, shape (N_kept, 9), dtype float32
        Columns: [xn, yn, zn, tn, un, vn, wn, Pn] (already normalized).
    snap_offsets : list[(t_phys, start_idx, end_idx)]
        Useful to slice rows belonging to one snapshot later.
    first_snap_raw : dict | None
        Untouched arrays for the first snapshot (validation plots).
    """
    if max_snapshots is not None:
        snaps = snaps[:max_snapshots]
    n_snaps = len(snaps)
    if n_snaps == 0:
        raise RuntimeError("No snapshots to load.")

    # Pre-allocate the OUTPUT tensor at upper bound. We'll trim at the end
    # if some snapshots had fewer valid points than N_POINTS_PER_SNAP.
    cap = n_snaps * n_points_per_snap
    data_np = np.empty((cap, 9), dtype=np.float32)
    snap_offsets = []
    write_ptr = 0

    col_map = None
    P_offset = None       # auto-detected (absolute vs gauge)
    first_snap_raw = None

    print(f"Loading {n_snaps} snapshots ({n_points_per_snap} pts each, "
          f"upper-bound RAM ~{cap * 9 * 4 / 1e6:.1f} MB)...")
    t0 = time.time()

    for i, (t_phys, path) in enumerate(snaps):
        try:
            data, col_map = load_snapshot(path, col_map=col_map)
        except Exception as exc:
            print(f"  [skip] {os.path.basename(path)}: {exc}")
            continue

        x, y, z = data["x"], data["y"], data["z"]
        u, v, w = data["u"], data["v"], data["w"]
        P       = data["P"]

        # Auto-detect pressure convention once (absolute vs gauge)
        if P_offset is None:
            if P.mean() > 1e4:
                P_offset = P_OP
                print(f"  Pressure mean={P.mean():.1f} Pa -> treating as absolute "
                      f"(subtract P_op={P_OP})")
            else:
                P_offset = 0.0
                print(f"  Pressure mean={P.mean():.3f} Pa -> treating as gauge")
        P = P - P_offset

        # Mask out points inside the cylinder (no fluid data there)
        inside = ((x - XC_CYL)**2 + (y - YC_CYL)**2 <= R_CYL**2) & (z <= H_CYL)
        valid_idx = np.where(~inside)[0]

        n_take = min(n_points_per_snap, len(valid_idx))
        picked = valid_idx[_subsample_indices(len(valid_idx), n_take, rng)]

        # Normalize directly into the pre-allocated slab
        end = write_ptr + n_take
        sl  = slice(write_ptr, end)
        data_np[sl, 0] = x[picked] / L_REF
        data_np[sl, 1] = y[picked] / L_REF
        data_np[sl, 2] = z[picked] / L_REF
        data_np[sl, 3] = t_phys / TIME_REF       # broadcast scalar
        data_np[sl, 4] = u[picked] / V_REF
        data_np[sl, 5] = v[picked] / V_REF
        data_np[sl, 6] = w[picked] / V_REF
        data_np[sl, 7] = P[picked] / P_REF

        snap_offsets.append((t_phys, write_ptr, end))
        write_ptr = end

        # Keep raw arrays of the first snapshot for downstream plots
        if keep_first_raw and first_snap_raw is None:
            first_snap_raw = dict(
                x=x.copy(), y=y.copy(), z=z.copy(),
                u=u.copy(), v=v.copy(), w=w.copy(),
                P=P.copy(),
                t=float(t_phys),
            )

        # Free the heaviest arrays held by `data` before the next iteration
        del data, x, y, z, u, v, w, P, valid_idx, picked

        if (i + 1) % verbose_every == 0 or i == 0 or i == n_snaps - 1:
            print(f"  [{i+1:4d}/{n_snaps}] t={t_phys:.4f}s  "
                  f"+{n_take} pts  (total={write_ptr:,})")

    # Trim unused tail (in case some snaps had < n_points_per_snap valid pts)
    data_np = data_np[:write_ptr]

    print(f"\nDone. Total supervised points: {len(data_np):,} "
          f"across {len(snap_offsets)} snapshots")
    print(f"Wall time: {time.time() - t0:.1f}s    "
          f"Memory: {data_np.nbytes / 1e6:.1f} MB")
    return data_np, snap_offsets, first_snap_raw


# ----- run loader -----
data_np, snap_offsets, first_snap_raw = load_all_snapshots(snaps)
n_snaps = len(snap_offsets)

## **Cell 06 - Train/Validation Split**
validação de treino e teste (todo 10° snapshot vira validação - se houver snapshots suficientes para isso).
- 10% de pontos para validação

- utilização de hold-out temporal: é um método de avaliação de modelos de aprendizado de máquina e previsões, essencial quando os dados possuem uma ordem cronológica (séries temporais). Ao contrário da validação cruzada tradicional, que mistura os dados aleatoriamente, o hold-out temporal divide os dados com base no tempo. Os dados mais antigos são usados para treinar o modelo, enquanto os dados mais recentes são separados (hold out) para testar o desempenho. Dessa forma, impede que os modelos aprendam somente informações do futuro para resolver informações do passado.

<font color="orange"> nós achamos que podemos tirar a terceira validação da máscara. </font>  


In [ ]:
# ============================================================
# CELL 6 — TRAIN / VALIDATION SPLIT (HOLD-OUT TEMPORAL)
# ============================================================
# We hold out ENTIRE timesteps (not random rows). This makes the validation
# loss an honest test of temporal generalization, not just spatial fitting.

VAL_EVERY      = 10    # every 10th snapshot index becomes validation
SKIP_FIRST     = False  # keep snapshot index 0 in train (often used as IC)

# Auto-relax VAL_EVERY if we don't have enough snapshots, so you actually get
# *some* validation points instead of an empty val set.
if n_snaps < 2 * VAL_EVERY:
    VAL_EVERY = max(2, n_snaps // 5)
    print(f"[info] only {n_snaps} snapshots -> reducing VAL_EVERY to {VAL_EVERY}")

val_snap_indices = set(range(0, n_snaps, VAL_EVERY))
if SKIP_FIRST:
    val_snap_indices.discard(0)

# Build a boolean mask over ALL rows from snap_offsets (no float-equality games)
val_mask = np.zeros(len(data_np), dtype=bool)
for i, (_t_phys, start, end) in enumerate(snap_offsets):
    if i in val_snap_indices:
        val_mask[start:end] = True

train_np = data_np[~val_mask]
val_np   = data_np[val_mask]

val_times_phys = [snap_offsets[i][0] for i in sorted(val_snap_indices)]
print(f"Train points: {len(train_np):,}    Val points: {len(val_np):,}")
print(f"Validation snapshots: {len(val_snap_indices)} held out")
print(f"  Physical times (s): "
      f"{[f'{t:.3f}' for t in val_times_phys[:5]]}"
      f"{' ...' if len(val_times_phys) > 5 else ''}")

## **Cell 07 - Collocation Points (domínio em que vamos avaliar os resíduos físicos)**
- número de pontos de colocação escolhidos - 100.000 pontos de colocação
- <font color="orange"> rever essa célula novamente pois konrad&thiago mudaram a forma que a colocação dos pontos funciona. </font>
- mudar a altura de Z


In [ ]:
# ============================================================
# CELL 7 — COLLOCATION POINTS (TRAPEZOID + TRIANGLE)
# ============================================================
# Two-stage refinement region:
#   Stage 1 (trapezoid):  cylinder + immediate near-wake (~3D)
#                          contains 75% of refinement points
#   Stage 2 (triangle):   developing wake (3D to ~8D)
#                          contains 25% of refinement points
# This concentrates 3x more density where gradients are strongest
# while still covering the full wake region.

N_COLL = 100_000

# --- Trapezoid geometry (stage 1: cylinder + immediate wake) ---
TRAP_HALF_WIDTH = 50.0       # full width = 100m (1.85D)
TRAP_Y_START    = 170.0      # base behind cylinder
TRAP_Y_END      = 320.0      # ~3D after cylinder center
TRAP_Z_MIN      = 0.0
TRAP_Z_MAX      = 2.0 * H_CYL   # 130m

# --- Triangle geometry (stage 2: developing wake) ---
TRI_HALF_WIDTH_BASE = TRAP_HALF_WIDTH   # continuous with trapezoid
TRI_Y_BASE          = TRAP_Y_END        # starts where trapezoid ends
TRI_Y_APEX          = 600.0             # 100m before outlet
TRI_Z_MIN           = TRAP_Z_MIN
TRI_Z_MAX           = TRAP_Z_MAX

# Trapezoid vertices (XY plane, projected through Z)
TRAP_V1 = (XC_CYL - TRAP_HALF_WIDTH, TRAP_Y_START)
TRAP_V2 = (XC_CYL + TRAP_HALF_WIDTH, TRAP_Y_START)
TRAP_V3 = (XC_CYL + TRAP_HALF_WIDTH, TRAP_Y_END)
TRAP_V4 = (XC_CYL - TRAP_HALF_WIDTH, TRAP_Y_END)

# Triangle vertices
TRI_V1 = (XC_CYL - TRI_HALF_WIDTH_BASE, TRI_Y_BASE)
TRI_V2 = (XC_CYL + TRI_HALF_WIDTH_BASE, TRI_Y_BASE)
TRI_V3 = (XC_CYL, TRI_Y_APEX)

# --- Allocation ---
N_REFINE     = 70_000           # total refinement points
N_TRAP       = int(0.75 * N_REFINE)   # 52,500 in trapezoid
N_TRI        = N_REFINE - N_TRAP       # 17,500 in triangle
N_FREESTREAM = 25_000
N_GROUND     = 5_000

assert N_TRAP + N_TRI + N_FREESTREAM + N_GROUND == N_COLL


def is_inside_cylinder(x, y, z):
    r2 = (x - XC_CYL)**2 + (y - YC_CYL)**2
    return (r2 <= R_CYL**2) & (z <= H_CYL)


def is_inside_trapezoid(x, y, z):
    """Rectangular box in XY (since trap is actually a rectangle here)."""
    return ((x >= TRAP_V1[0]) & (x <= TRAP_V2[0]) &
            (y >= TRAP_Y_START) & (y <= TRAP_Y_END) &
            (z >= TRAP_Z_MIN) & (z <= TRAP_Z_MAX))


def is_inside_triangle(x, y, v1, v2, v3):
    """Barycentric test."""
    x1, y1 = v1
    x2, y2 = v2
    x3, y3 = v3
    denom = (y2 - y3) * (x1 - x3) + (x3 - x2) * (y1 - y3)
    a = ((y2 - y3) * (x - x3) + (x3 - x2) * (y - y3)) / denom
    b = ((y3 - y1) * (x - x3) + (x1 - x3) * (y - y3)) / denom
    c = 1.0 - a - b
    return (a >= 0) & (b >= 0) & (c >= 0)


def is_inside_triangle_z(x, y, z):
    in_xy = is_inside_triangle(x, y, TRI_V1, TRI_V2, TRI_V3)
    in_z  = (z >= TRI_Z_MIN) & (z <= TRI_Z_MAX)
    return in_xy & in_z


def sample_trapezoid_pts(n):
    """Sample n points inside trapezoid (rectangle here), excluding cylinder."""
    pts = []
    while len(pts) < n:
        n_try = int((n - len(pts)) * 1.5) + 100
        x = np.random.uniform(XC_CYL - TRAP_HALF_WIDTH, XC_CYL + TRAP_HALF_WIDTH, n_try)
        y = np.random.uniform(TRAP_Y_START, TRAP_Y_END, n_try)
        z = np.random.uniform(TRAP_Z_MIN, TRAP_Z_MAX, n_try)

        keep = ~is_inside_cylinder(x, y, z)
        x = x[keep]; y = y[keep]; z = z[keep]
        pts.extend(zip(x, y, z))

    return np.array(pts[:n], dtype=np.float32)


def sample_triangle_pts(n):
    """Sample n points inside triangle prism, excluding cylinder (just in case)."""
    pts = []
    x_min = min(TRI_V1[0], TRI_V2[0], TRI_V3[0])
    x_max = max(TRI_V1[0], TRI_V2[0], TRI_V3[0])
    y_min = min(TRI_V1[1], TRI_V2[1], TRI_V3[1])
    y_max = max(TRI_V1[1], TRI_V2[1], TRI_V3[1])

    while len(pts) < n:
        n_try = int((n - len(pts)) * 2.5) + 100
        x = np.random.uniform(x_min, x_max, n_try)
        y = np.random.uniform(y_min, y_max, n_try)
        z = np.random.uniform(TRI_Z_MIN, TRI_Z_MAX, n_try)

        in_tri = is_inside_triangle(x, y, TRI_V1, TRI_V2, TRI_V3)
        out_cyl = ~is_inside_cylinder(x, y, z)
        keep = in_tri & out_cyl

        x = x[keep]; y = y[keep]; z = z[keep]
        pts.extend(zip(x, y, z))

    return np.array(pts[:n], dtype=np.float32)


def sample_freestream_pts(n):
    """Sample n points OUTSIDE both trapezoid and triangle, excluding cylinder."""
    pts = []
    while len(pts) < n:
        n_try = int((n - len(pts)) * 1.5) + 100
        x = np.random.uniform(X_MIN, X_MAX, n_try)
        y = np.random.uniform(Y_MIN, Y_MAX, n_try)
        z = np.random.uniform(Z_MIN, Z_MAX, n_try)

        out_trap = ~is_inside_trapezoid(x, y, z)
        out_tri  = ~is_inside_triangle_z(x, y, z)
        out_cyl  = ~is_inside_cylinder(x, y, z)
        keep = out_trap & out_tri & out_cyl

        x = x[keep]; y = y[keep]; z = z[keep]
        pts.extend(zip(x, y, z))

    return np.array(pts[:n], dtype=np.float32)


def sample_ground_pts(n):
    """Sample n points in ground boundary layer (z < 5m)."""
    pts = []
    while len(pts) < n:
        n_try = int((n - len(pts)) * 1.5) + 100
        x = np.random.uniform(X_MIN, X_MAX, n_try)
        y = np.random.uniform(Y_MIN, Y_MAX, n_try)
        z = np.random.uniform(0.0, 5.0, n_try)

        keep = ~is_inside_cylinder(x, y, z)
        x = x[keep]; y = y[keep]; z = z[keep]
        pts.extend(zip(x, y, z))

    return np.array(pts[:n], dtype=np.float32)


def sample_collocation(n=N_COLL):
    trap_pts = sample_trapezoid_pts(N_TRAP)
    tri_pts  = sample_triangle_pts(N_TRI)
    free_pts = sample_freestream_pts(N_FREESTREAM)
    grnd_pts = sample_ground_pts(N_GROUND)

    xyz = np.concatenate([trap_pts, tri_pts, free_pts, grnd_pts], axis=0)
    np.random.shuffle(xyz)

    t = np.random.uniform(0.0, 1.0, len(xyz)).astype(np.float32).reshape(-1, 1)
    xyz_n = xyz / L_REF

    return np.concatenate([xyz_n, t], axis=1).astype(np.float32)


# --- Generate ---
coll_np = sample_collocation(N_COLL)

print(f"Collocation points: {len(coll_np):,}")
print(f"  Trapezoid (cylinder + near-wake):  {N_TRAP:,}  ({100*N_TRAP/N_COLL:.0f}%)")
print(f"  Triangle (developing wake):        {N_TRI:,}  ({100*N_TRI/N_COLL:.0f}%)")
print(f"  Free-stream:                       {N_FREESTREAM:,}  ({100*N_FREESTREAM/N_COLL:.0f}%)")
print(f"  Ground layer:                      {N_GROUND:,}  ({100*N_GROUND/N_COLL:.0f}%)")

print(f"\nTrapezoid:  y=[{TRAP_Y_START:.0f}, {TRAP_Y_END:.0f}] | "
      f"width={2*TRAP_HALF_WIDTH:.0f}m ({2*TRAP_HALF_WIDTH/(2*R_CYL):.2f}D)")
print(f"Triangle:   y=[{TRI_Y_BASE:.0f}, {TRI_Y_APEX:.0f}] | "
      f"base width={2*TRI_HALF_WIDTH_BASE:.0f}m | apex at y={TRI_Y_APEX:.0f}")

# --- Density check ---
vol_trap = (2*TRAP_HALF_WIDTH) * (TRAP_Y_END - TRAP_Y_START) * (TRAP_Z_MAX - TRAP_Z_MIN)
vol_tri  = 0.5 * (2*TRI_HALF_WIDTH_BASE) * (TRI_Y_APEX - TRI_Y_BASE) * (TRI_Z_MAX - TRI_Z_MIN)
vol_total = (X_MAX-X_MIN) * (Y_MAX-Y_MIN) * (Z_MAX-Z_MIN)
vol_free = vol_total - vol_trap - vol_tri

print(f"\nDensity analysis (pts per 1000 m³):")
print(f"  Trapezoid:   {1000*N_TRAP/vol_trap:.1f}")
print(f"  Triangle:    {1000*N_TRI/vol_tri:.1f}")
print(f"  Free-stream: {1000*N_FREESTREAM/vol_free:.2f}")


# --- Visualization ---
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Circle, Rectangle

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# XY view (top-down)
ax = axes[0]
ax.scatter(coll_np[:, 0] * L_REF, coll_np[:, 1] * L_REF,
            s=0.5, alpha=0.3, c='steelblue', rasterized=True)

# Cylinder
ax.add_patch(Circle((XC_CYL, YC_CYL), R_CYL, fill=False, color='red', lw=2, zorder=5))

# Trapezoid (rectangle here)
ax.add_patch(Rectangle((XC_CYL - TRAP_HALF_WIDTH, TRAP_Y_START),
                         2*TRAP_HALF_WIDTH, TRAP_Y_END - TRAP_Y_START,
                         fill=False, color='green', lw=2, ls='--', zorder=4,
                         label=f'Trapezoid ({N_TRAP/1000:.0f}k pts)'))

# Triangle
ax.add_patch(Polygon([TRI_V1, TRI_V2, TRI_V3],
                       fill=False, color='orange', lw=2, ls='--', zorder=4,
                       label=f'Triangle ({N_TRI/1000:.0f}k pts)'))

# Flow arrow
ax.annotate('', xy=(20, 120), xytext=(20, 30),
             arrowprops=dict(arrowstyle='->', color='gray', lw=2))
ax.text(25, 75, 'flow', fontsize=10, color='gray')

ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')
ax.set_title('Top view (XY)\nRed: cylinder | Green: trapezoid (75%) | Orange: triangle (25%)')
ax.set_xlim(X_MIN, X_MAX)
ax.set_ylim(Y_MIN, Y_MAX)
ax.set_aspect('equal')
ax.legend(loc='upper right', fontsize=9)

# XZ view (side)
ax = axes[1]
ax.scatter(coll_np[:, 0] * L_REF, coll_np[:, 2] * L_REF,
            s=0.5, alpha=0.3, c='steelblue', rasterized=True)
ax.add_patch(Rectangle((XC_CYL - R_CYL, 0), 2*R_CYL, H_CYL,
                         fill=False, color='red', lw=2, zorder=5))
ax.axhline(y=TRAP_Z_MAX, color='orange', ls='--', lw=1, alpha=0.5)
ax.text(10, TRAP_Z_MAX + 3, f'Refinement Z_max = {TRAP_Z_MAX:.0f}m',
         fontsize=9, color='orange')
ax.set_xlabel('x [m]')
ax.set_ylabel('z [m]')
ax.set_title('Side view (XZ)\nRed: cylinder')
ax.set_xlim(X_MIN, X_MAX)
ax.set_ylim(Z_MIN, Z_MAX)

plt.tight_layout()
plt.savefig('collocation_distribution.png', dpi=120)
plt.show()

# Quantos pontos estão GENUINAMENTE dentro do cilindro (3D)?
xyz_phys = coll_np[:, :3] * L_REF
inside_3d = is_inside_cylinder(xyz_phys[:, 0], xyz_phys[:, 1], xyz_phys[:, 2])
print(f"Pontos dentro do cilindro 3D: {inside_3d.sum()}")
print(f"(Esperado: 0)")

# Quantos pontos aparecem "dentro" do círculo no plot XY (mas não no 3D)?
r2_proj = (xyz_phys[:, 0] - XC_CYL)**2 + (xyz_phys[:, 1] - YC_CYL)**2
inside_xy_proj = r2_proj <= R_CYL**2
print(f"Pontos com projeção XY dentro do cilindro: {inside_xy_proj.sum()}")
print(f"(Inclui pontos acima do cilindro, em z > {H_CYL}m)")

# Desses, quantos estão acima do topo do cilindro?
above_top = inside_xy_proj & (xyz_phys[:, 2] > H_CYL)
print(f"Pontos acima do topo: {above_top.sum()}")

## **Cell 08 - Neural Network**

criação da função que implementa a camada de hard constraining e a função que inicializa a PINN.

- criação da layer que vai atuar como hard constraining. <font color="orange"> revisar essa parte porque as coisas mudaram muito!! </font>

- inicialização da rede neural:
  - função de ativação: TanH;
  - inicialização do kernel com glorot_uniform (como os pesos daquela camada neural serão inicializados antes do treinamento: `glorot_uniform` é a implementação do método de inicialização dos pesos com distribuição uniforme de Xavier Glorot e Yoshua Bengio para evitar explosão de gradientes e gradient vanishing- Xavier Inicialization).
  - inicialização dos bias no início do tratamento com zeros.
- arquitetura - modelo funcional:
  - uma camada de input;
  - oito camadas densas com 256 neurônios; <font color="pink"> a yasmin e a taynah vão implementar os blocos residuais aqui. </font>
  - uma camada de saída parcial com 5 neurônios;
  - uma camada personalizada para aplicação da hard constraining (o parâmetro é o controle da "espessura aparente" da camada limite térmica);

- nome da variável: `model`

In [ ]:
#tentativa de criação dos blocos residuais

class DenseResidualBlock(layers.Layer): #o argumento passado é a classe pai, da qual a nossa classe vai herdar

    def __init__(self, units, activation='tanh',
                 kernel_initializer='glorot_uniform', **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.activation_fn = tf.keras.activations.get(activation)
        self.kernel_initializer = kernel_initializer

    def build(self, input_shape):
        self.dense1 = tf.keras.layers.Dense(
            self.units, kernel_initializer=self.kernel_initializer
        )
        self.dense2 = tf.keras.layers.Dense(
            self.units, kernel_initializer=self.kernel_initializer
        )
        if input_shape[-1] != self.units: #usa projeção linear no atalho se as dimensões forem diferentes para poder fazer a soma dos elementos entre os tensores.
            self.projection = tf.keras.layers.Dense(
                self.units, use_bias=False,
                kernel_initializer=self.kernel_initializer
            )
        else:
            self.projection = None
        super().build(input_shape)

    def call(self, inputs):
        h = self.dense1(inputs)
        h = self.activation_fn(h)
        h = self.dense2(h)
        shortcut = self.projection(inputs) if self.projection is not None else inputs
        h = h + shortcut
        h = self.activation_fn(h)
        return h

In [ ]:
# ============================================================
# CELL 9 — NEURAL NETWORK (Keras functional)
# ============================================================

class TanH(layers.Layer):
    def call(self, x):
        return tf.tanh(x)

class HardConstraintLayer(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    @staticmethod
    def smooth_max(a, b, k=5000.0):
        m = tf.maximum(a, b)
        return m + (1.0 / k) * tf.math.log(
            tf.exp(k * (a - m)) + tf.exp(k * (b - m))
        )

    def call(self, inputs):
        xyzt, raw_pred = inputs

        xn = xyzt[:, 0:1]
        yn = xyzt[:, 1:2]
        zn = xyzt[:, 2:3]

        un = raw_pred[:, 0:1]
        vn = raw_pred[:, 1:2]
        wn = raw_pred[:, 2:3]
        Pn = raw_pred[:, 3:4]

        # --- Constantes (idênticas às do meshgrid) ---
        mt       = 5600.0
        rate     = 3300.0
        k_smooth = 5000.0

        # --- 1. Funções de distância (cada uma = 0 SOBRE seu plano) ---
        D_inlet  = tf.math.tanh(mt * (yn - YN_MIN))      # 0 em y = Y_MIN  (entrada)
        D_outlet = tf.math.tanh(mt * (YN_MAX - yn))      # 0 em y = Y_MAX  (saída)  ← NOVO
        D_ground = tf.math.tanh(mt * (zn - ZN_MIN))      # 0 em z = Z_MIN  (chão)
        D_xmin   = tf.math.tanh(mt * (xn - XN_MIN))      # 0 em x = X_MIN
        D_xmax   = tf.math.tanh(mt * (XN_MAX - xn))      # 0 em x = X_MAX
        D_ztop   = tf.math.tanh(mt * (ZN_MAX - zn))      # 0 em z = Z_MAX

        # Distância 3D ao cilindro finito (lateral + topo)
        r2       = tf.square(xn - XC_N) + tf.square(yn - YC_N)
        d_radial = tf.sqrt(r2 + 1e-12) - R_N
        d_top    = zn - H_N
        d_cyl_3d = self.smooth_max(d_radial, d_top, k=k_smooth)
        D_cyl    = tf.math.tanh(mt * tf.maximum(d_cyl_3d, 0.0))

        # --- 2. Multiplicador único para u, v, w (zero em toda fronteira Dirichlet) ---
        # Observação: D_outlet NÃO entra aqui — a saída não tem velocidade nem
        # temperatura prescritas, apenas pressão (tratada separadamente abaixo).
        D_vel = D_inlet * D_ground * D_cyl * D_xmin * D_xmax * D_ztop

        # --- 3. Decays freestream (fuzzy-OR sobre entrada + 2 laterais + topo) ---
        decay_inlet = tf.exp(-rate * (yn - YN_MIN))
        decay_xmin  = tf.exp(-rate * (xn - XN_MIN))
        decay_xmax  = tf.exp(-rate * (XN_MAX - xn))
        decay_ztop  = tf.exp(-rate * (ZN_MAX - zn))
        decay_freestream = 1.0 - (1.0 - decay_inlet) \
                               * (1.0 - decay_xmin)  \
                               * (1.0 - decay_xmax)  \
                               * (1.0 - decay_ztop)

        # --- 4. Velocidade hard ansatz ---
        v_baseline = VN_INLET * decay_freestream * D_ground * D_cyl
        u_hard = un * D_vel
        v_hard = vn * D_vel + v_baseline
        w_hard = wn * D_vel

        # --- 5. Pressão hard ansatz: P = 0 em y = Y_MAX (saída) ---
        # Sem baseline (o valor prescrito é 0), então basta multiplicar pela distância.
        P_hard = Pn * D_outlet

        return tf.concat([u_hard, v_hard, w_hard, P_hard], axis=1)


def build_pinn(n_blocks=4, n_neurons=256, t_sharp=200.0):
    inp = layers.Input(shape=(4,), name="xyzt")

    # --- Projeção da entrada: (N, 4) -> (N, n_neurons) ---
    h = layers.Dense(n_neurons,
                     kernel_initializer="glorot_uniform",
                     bias_initializer="zeros",
                     name="input_proj")(inp)
    h = TanH(name="input_proj_tanh")(h)

    # --- Torre residual ---
    for i in range(n_blocks):
        h = DenseResidualBlock(n_neurons,
                               activation='tanh',
                               kernel_initializer='glorot_uniform',
                               name=f"res_block_{i}")(h)

    # --- Saída raw e hard constraints (idênticos ao original) ---
    raw_out = layers.Dense(5, name="raw_fields")(h)
    final_out = HardConstraintLayer(name="hard_constraints")([inp, raw_out])

    return Model(inp, final_out, name="PINN_residual")


MODEL_BLOCKS  = 4      # antes era MODEL_HIDDEN = 8
MODEL_NEURONS = 256
T_SHARP       = 200.0

model = build_pinn(MODEL_BLOCKS, MODEL_NEURONS, t_sharp=T_SHARP)

In [ ]:
model.summary()  # confere shapes e contagem de params

# Sanity check com pontos fake
xyzt_test = tf.random.uniform((100, 4), minval=0.0, maxval=1.0)
out = model(xyzt_test)
print(out.shape)  # (100, 4) — saída do HardConstraintLayer (u, v, w, P)

## **Cell 09 - Physics Residuals (TF GradientTape)**
<font color="red"> rever toda essa célula porque o konrad mandou um novo código pelo whatsapp. </font>

-  definição das constantes como variáveis internas (convenção do underline) - tensores no tf;
- criação das escalas residuais (para normalização dos resíduos ao final);

- definição da função dos resíduos físicos;
  - cálculo automático das derivadas pelo GradientTape (diferenciação automática, que percorre o grafo de operações gravadas de trás para frente aplicando a regra da cadeia);
    - tape 1 calcula a primeira derivada e tape2 calcula a segunda derivada - derivada da primeira derivada (NV têm termos difusivos - laplacianos);
    - o termo `persistent=True` permite usar o mesmo tape múltiplas vezes (uma vez para cada campo);
    - função `tape.watch()`: diz ao tape quais tensores ele deve monitorar. como os pontos de colocação são constantes/tensores de entrada comuns, não variáveis treinadas, o tape2 ignoraria os coll_pts sem essa função e não derivaria;

  - predição das variáveis de saída com o trecho `pred = model(coll_pts, training=True)`
  - desnormalização para as unidades físicas;
  - densidade pelo gás ideial incompressível: densidade dad;  
  - cálculo das primeiras derivadas em relação a cada uma das saídas. cada uma das derivadas tem como retorno as derivadas em [x, y, z, t] <font color="orange"> dúvida: essa conversão das coordenadas normalizadas para as físicas pela regra da cadeia está certa?? </font>
  - `u_x` e os outros desempacota o vetor das derivadas em variáveis nomeadas para facilitar a leitura das equações a seguir;
  - `del tape1` libera a memória pois tape1 não é mais necessário depois que todas as primeiras derivadas foram calculadas.
  
  - definição da função dos laplacianos:  
    - `tape2.gradient` calcula as segundas derivadas e divide por _L para converte as coordenadas normalizadas para coordenadas físicas. <font color="orange"> dúvida: essa derivada está sendo calculada corretamente? </font>  
  
  - `div_V = u_x + v_y + w_z`: cálculo da divergência da velocidade. <font color="orange">tá certo. </font>

  - equações da continuidade (*ver!*) e de momentum (navier-stokes nos três eixos, sendo que no eixo z tem o empuxo - boussinesq) - a de energia foi retirada pois a temperatura não é mais uma entrada.

  - retorno dos resíduos normalizados pelas escalas (escalonamento dos resíduos para torná-los da mesma categoria).

In [ ]:
# ============================================================
# CELL 10 — PHYSICS RESIDUALS (TF GradientTape)
# ============================================================
# TF derivatives require nested GradientTape for Laplacians.
# We compute everything in physical units inside the residuals, then scale the loss.

# Pre-compute constants as tf tensors to avoid re-creation
_L  = tf.constant(L_REF,  tf.float32)
_V  = tf.constant(V_REF,  tf.float32)
_P  = tf.constant(P_REF,  tf.float32)
_tR = tf.constant(TIME_REF, tf.float32)
_RHO_INF = tf.constant(RHO_INF, tf.float32)
_P_OP    = tf.constant(P_OP,    tf.float32)
_R_SP    = tf.constant(R_SP,    tf.float32)
_T_INF   = tf.constant(T_INF,   tf.float32)
_MU      = tf.constant(MU,      tf.float32)
_G       = tf.constant(G_ACC,   tf.float32)

# Residual scales (to normalize loss contributions)
SCALE_CONT = RHO_INF * V_REF / L_REF
SCALE_MOM  = RHO_INF * V_REF**2 / L_REF

_SCALE_CONT = tf.constant(SCALE_CONT, tf.float32)
_SCALE_MOM  = tf.constant(SCALE_MOM,  tf.float32)

def physics_residuals(coll_pts, model):
    """
    Compute PDE residuals at normalized collocation points (N, 4).
    Uses nested GradientTape for second derivatives (Laplacians).
    Returns scaled residuals ready for MSE loss.
    """
    with tf.GradientTape(persistent=True) as tape2:
        tape2.watch(coll_pts)
        with tf.GradientTape(persistent=True) as tape1:
            tape1.watch(coll_pts)
            pred = model(coll_pts, training=True)    # (N, 4) normalized
            un = pred[:, 0:1]; vn = pred[:, 1:2]; wn = pred[:, 2:3]
            Pn = pred[:, 3:4]
            # Denormalize
            u = un * _V
            v = vn * _V
            w = wn * _V
            P = Pn * _P
            # Density from ideal gas
            rho = _RHO_INF

        # First derivatives of each field w.r.t. (xn, yn, zn, tn)
        # Divide by L_REF or TIME_REF to convert to physical derivatives
        du = tape1.gradient(u, coll_pts) / tf.stack([_L, _L, _L, _tR])
        dv = tape1.gradient(v, coll_pts) / tf.stack([_L, _L, _L, _tR])
        dw = tape1.gradient(w, coll_pts) / tf.stack([_L, _L, _L, _tR])
        dP = tape1.gradient(P, coll_pts) / tf.stack([_L, _L, _L, _tR])

        u_x, u_y, u_z, u_t = du[:, 0], du[:, 1], du[:, 2], du[:, 3]
        v_x, v_y, v_z, v_t = dv[:, 0], dv[:, 1], dv[:, 2], dv[:, 3]
        w_x, w_y, w_z, w_t = dw[:, 0], dw[:, 1], dw[:, 2], dw[:, 3]
        P_x, P_y, P_z      = dP[:, 0], dP[:, 1], dP[:, 2]


        del tape1

    # Second derivatives for Laplacians
    def lap(fx, fy, fz):
        fxx = tape2.gradient(fx, coll_pts)[:, 0] / _L
        fyy = tape2.gradient(fy, coll_pts)[:, 1] / _L
        fzz = tape2.gradient(fz, coll_pts)[:, 2] / _L
        return fxx + fyy + fzz

    lap_u = lap(u_x, u_y, u_z)
    lap_v = lap(v_x, v_y, v_z)
    lap_w = lap(w_x, w_y, w_z)

    del tape2

    # Flatten tensors to 1D
    u_flat = tf.squeeze(u, -1);   v_flat = tf.squeeze(v, -1);   w_flat = tf.squeeze(w, -1)
    rho_f  = tf.squeeze(rho, -1)

    div_V = u_x + v_y + w_z

    # Continuity: ∂ρ/∂t + V·∇ρ + ρ∇·V
    r_cont = div_V

    # Momentum equations
    r_mu = rho_f * (u_t + u_flat*u_x + v_flat*u_y + w_flat*u_z) + P_x - _MU*lap_u
    r_mv = rho_f * (v_t + u_flat*v_x + v_flat*v_y + w_flat*v_z) + P_y - _MU*lap_v
    r_mw = rho_f * (w_t + u_flat*w_x + v_flat*w_y + w_flat*w_z) + P_z - _MU*lap_w

    # Scaled residuals (dimensionless ~ O(1))
    return (r_cont / _SCALE_CONT,
            r_mu   / _SCALE_MOM,
            r_mv   / _SCALE_MOM,
            r_mw   / _SCALE_MOM)

## **Cell 10 - Loss Functions**
- definição da função de erro: cálculo do erro quadrático médio, resultando em um escalar para averiguar a perda da rede neural.
  - por que erro quadrático médio (por que quadrático e não absoluto)? para penalizar mais erros grandes e ser diferenciável em todos os pontos.
- definição da função de perda: <font color="orange">dúvida: o que é o batch? qual o tamanho dele e onde é definido? - celula 13/14</font>
  - faz o erro quadrático médio entre os valores reais (tgt) e os valores previstos pelo modelo (através da função `model`e guardados na variável `pred`). retorna esse erro. <font color="orange"> poderíamos colocar direto a função `tf.reduce_mean(tf.square(x))`, não? não. </font>

In [ ]:
# ============================================================
# CELL 11 — LOSS FUNCTIONS
# ============================================================

def mse(x):
    return tf.reduce_mean(tf.square(x))

def data_loss(batch, model):
    xyzt = batch[:, :4] # pega as primeiras quatro colunas - entradas do modelo (xyzt)
    tgt  = batch[:, 4:] # pega as últimas quatro colunas (o que sobrou) - valores alvo (por isso o nome tgt - de target) (u,v,w,P)
    pred = model(xyzt, training=True)
    return mse(pred - tgt)

def physics_loss(coll_pts, model):
    r_c, r_mu, r_mv, r_mw = physics_residuals(coll_pts, model)
    return mse(r_c), mse(r_mu), mse(r_mv), mse(r_mw)

## **Cell 11 - Loss Weights**

<font color="orange">dicionário com os pesos que será usado no loop de treinamento. confirmar porque só alguns são inicializados (e os de BC não são). </font>

In [ ]:
# ============================================================
# CELL 12 — LOSS WEIGHTS
# ============================================================

W = {
    "data":     50.0,
    "cont":      0.5,
    "mom_u":     1.0,
    "mom_v":     1.0,
    "mom_w":     1.0,
}

for k, v in W.items():
    print(f"  W[{k:10s}] = {v}")

## **Cell 12 - Prepare Tensors for Training**
-

In [ ]:
# ============================================================
# CELL 13 — PREPARE TENSORS FOR TRAINING
# ============================================================
# Two-stage temporal shuffling strategy:
#   1) Pre-shuffle the ORDER of snapshots in train_np (zero RAM cost).
#      This guarantees that consecutive points in memory come from
#      randomly distinct timesteps.
#   2) Use tf.data.Dataset.shuffle on top, for intra-batch mixing.
# Combined effect: every batch contains points from many distinct
# timesteps (initialization, established, wake), giving balanced
# gradients across all flow regimes.

BATCH_SIZE = 1024

# --- Stage 1: pre-shuffle the snapshot order in train_np ---
# We currently have train_np ordered as [snap_a points, snap_b points, ...]
# where (a, b, ...) is the training snapshot order. We rebuild train_np
# with a random permutation of that order.

rng = np.random.default_rng(seed=SEED)

# Reconstruct training snapshot indices (those NOT in val)
train_snap_indices_ordered = [i for i in range(n_snaps) if i not in val_snap_indices]

# Random permutation of the order they appear in train_np
shuffled_snap_order = rng.permutation(train_snap_indices_ordered)

# Reassemble train_np with permuted snapshot order
train_np_pieces = []
for i in shuffled_snap_order:
    _, start, end = snap_offsets[i]
    train_np_pieces.append(data_np[start:end])
train_np = np.concatenate(train_np_pieces, axis=0)

print(f"Snapshots in train (first 10 in new order): {shuffled_snap_order[:10].tolist()}")
print(f"train_np shape after pre-shuffle: {train_np.shape}")

# --- Stage 2: tf.data.Dataset with intra-batch shuffling ---
train_tf = tf.data.Dataset.from_tensor_slices(train_np)
train_tf = train_tf.shuffle(
    buffer_size=min(len(train_np), 1_000_000),  # 1M is enough given pre-shuffle
    seed=SEED,
    reshuffle_each_iteration=True,
)
train_tf = train_tf.batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.AUTOTUNE)

# --- Other tensors unchanged ---
val_tensor  = tf.constant(val_np,  dtype=tf.float32)
coll_tensor = tf.Variable(coll_np, dtype=tf.float32)  # resampleable

steps_per_epoch = int(np.ceil(len(train_np) / BATCH_SIZE))
print(f"Batch size: {BATCH_SIZE}   Steps/epoch: {steps_per_epoch}")


# --- Sanity check: verify temporal mixing in a sample batch ---
sample_batch = next(iter(train_tf))
t_values = sample_batch[:, 3].numpy()
print(f"\nTemporal mixing diagnostic (first batch):")
print(f"  Unique timesteps in batch: {len(np.unique(t_values))} / {len(train_snap_indices_ordered)} train snapshots")
print(f"  Range of t (normalized): [{t_values.min():.4f}, {t_values.max():.4f}]")
print(f"  (good: many unique values, range covering most of [0, 1])")

#thiago mudou!!!

In [ ]:
# ============================================================
# CELL 14 — TRAINING STEP (compiled)
# ============================================================

import tensorflow as tf
tf.config.run_functions_eagerly(False)
tf.data.experimental.enable_debug_mode()  # opcional, deixa erros mais legíveis

N_COLL_PER_STEP = 1024

optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)

@tf.function(jit_compile=True)
def train_step(batch, coll_sub, bc_d, w_data, w_c, w_mu, w_mv, w_mw, w_out):
    with tf.GradientTape() as tape:
        l_d = data_loss(batch, model)
        l_c, l_mu, l_mv, l_mw = physics_loss(coll_sub, model)

        total = (w_data * l_d
                  + w_c * l_c + w_mu * l_mu + w_mv * l_mv + w_mw * l_mw)

    grads = tape.gradient(total, model.trainable_variables)
    grads, _ = tf.clip_by_global_norm(grads, 1.0)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

    return total, l_d, l_c + l_mu + l_mv + l_mw

In [ ]:
# ============================================================
# CELL 15 — TRAINING LOOP (continuação de treino anterior)
# ============================================================

DEBUG_MODE = "full"   # "fast" | "medium" | "full"

if DEBUG_MODE == "fast":
    N_EPOCHS, LR_SCHEDULE = 5, []
    LOG_EVERY, SAVE_EVERY = 1, 999_999
elif DEBUG_MODE == "medium":
    N_EPOCHS    = 5
    LR_SCHEDULE = [(150, 5e-4), (350, 2e-4)]
    LOG_EVERY   = 10
    SAVE_EVERY  = 100
elif DEBUG_MODE == "full":
    N_EPOCHS    = 5   # menos epochs, RAM não vai explodir
    LR_SCHEDULE = [(1500, 5e-5), (3000, 2e-5)]   # decays garantidos
    LOG_EVERY   = 50
    SAVE_EVERY  = 500   # salva mais frequentemente

RESAMPLE_COLL_EVERY = 999_999
RESAMPLE_BC_EVERY   = 999_999

# --- Detecção de continuação ---
if 'history' not in globals() or len(history.get('epoch', [])) == 0:
    history = {"epoch": [], "total": [], "data": [], "phys": [],
                "bc": [], "val": [], "lr": []}
    epoch_offset = 0
    print("Iniciando treino do zero")
else:
    epoch_offset = max(history["epoch"])
    print(f"Continuando treino. Última época registrada: {epoch_offset}")

# CRÍTICO: NÃO RESETAR HISTORY AQUI! Removida linha problemática.

CHECKPOINT_DIR = "./pinn_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Convert weights to tf constants for @tf.function
w_data = tf.constant(W["data"],    tf.float32)
w_c    = tf.constant(W["cont"],    tf.float32)
w_mu   = tf.constant(W["mom_u"],   tf.float32)
w_mv   = tf.constant(W["mom_v"],   tf.float32)
w_mw   = tf.constant(W["mom_w"],   tf.float32)

print(f"\n{'='*70}")
print(f"  Training PINN: {N_EPOCHS} new epochs (continuing from ep {epoch_offset})")
print(f"  Total after: {epoch_offset + N_EPOCHS} epochs")
print(f"  LR atual: {float(optimizer.learning_rate.numpy()):.2e}")
print(f"  LR schedule: {LR_SCHEDULE}")
print(f"  Train points: {len(train_np):,}   Val points: {len(val_np):,}")
print(f"  Collocation: {N_COLL:,}   Physics batch per step: {N_COLL_PER_STEP}")
print(f"{'='*70}\n")

t0 = time.time()

for epoch in range(1, N_EPOCHS + 1):
    # --- Learning rate schedule ---
    for ep_thr, lr in LR_SCHEDULE:
        if epoch == ep_thr:
            optimizer.learning_rate.assign(lr)
            print(f"  [epoch {epoch}] LR → {lr}")

    # --- Resample collocation and BC points (desligado para esta rodada) ---
    if epoch % RESAMPLE_COLL_EVERY == 0:
        coll_tensor.assign(sample_collocation(N_COLL))
    if epoch % RESAMPLE_BC_EVERY == 0:
        new_bc = sample_bc()
        for k in bc_tensors:
            bc_tensors[k].assign(new_bc[k])

    # --- Training batches ---
    ep_total = ep_d = ep_phys = ep_bc = 0.0
    n_steps = 0
    for batch in train_tf:
        idx = tf.random.uniform((N_COLL_PER_STEP,), 0, N_COLL, dtype=tf.int32)
        coll_sub = tf.gather(coll_tensor, idx)

        total, l_d, l_phys, l_bc = train_step(batch, coll_sub, bc_tensors,
                                                w_data, w_c, w_mu, w_mv, w_mw,
                                                w_in, w_out, w_gnd, w_cs, w_ct)
        ep_total += float(total)
        ep_d     += float(l_d)
        ep_phys  += float(l_phys)
        ep_bc    += float(l_bc)
        n_steps  += 1

    ep_total /= n_steps
    ep_d     /= n_steps
    ep_phys  /= n_steps
    ep_bc    /= n_steps

    history["epoch"].append(epoch + epoch_offset)
    history["total"].append(ep_total)
    history["data"].append(ep_d)
    history["phys"].append(ep_phys)
    history["lr"].append(float(optimizer.learning_rate.numpy()))

    if epoch % LOG_EVERY == 0 or epoch == 1:
        val_pred = model(val_tensor[:, :4], training=False)
        val_mse  = float(tf.reduce_mean(tf.square(val_pred - val_tensor[:, 4:])))
        history["val"].append(val_mse)
        elapsed = time.time() - t0
        print(f"Ep {epoch:>5d}/{N_EPOCHS} | "
              f"total {ep_total:.3e} | data {ep_d:.3e} | "
              f"phys {ep_phys:.3e} | bc {ep_bc:.3e} | val {val_mse:.3e} | "
              f"lr {optimizer.learning_rate.numpy():.1e} | "
              f"{elapsed:.0f}s")

    if epoch % SAVE_EVERY == 0:
        # Inclui epoch_offset no nome do checkpoint para não sobrescrever os antigos
        ep_total_idx = epoch + epoch_offset
        path = os.path.join(CHECKPOINT_DIR, f"pinn_ep{ep_total_idx:06d}.weights.h5")
        model.save_weights(path)
        print(f"  checkpoint saved: {path}")

print(f"\nTraining complete: {time.time()-t0:.1f}s")
print(f"Total epochs trained: {epoch_offset + N_EPOCHS}")

In [ ]:
# ============================================================
# CELL 16 — SAVE FINAL MODEL + METADATA
# ============================================================

model.save_weights("pinn_final.weights.h5")

metadata = dict(
    L_ref=float(L_REF), V_ref=float(V_REF),
    P_ref=float(P_REF), time_ref=float(TIME_REF),
    X_min=X_MIN, X_max=X_MAX, Y_min=Y_MIN, Y_max=Y_MAX, Z_min=Z_MIN, Z_max=Z_MAX,
    D_cyl=D_CYL, H_cyl=H_CYL, xc_cyl=XC_CYL, yc_cyl=YC_CYL,
    V_inf=V_INF,
    P_op=P_OP, R_sp=R_SP, mu=MU, cp=CP, k_th=K_TH, g=G_ACC, rho_inf=float(RHO_INF),
    t_min=float(times_all[0]), t_max=float(times_all[-1]), n_snaps=int(n_snaps),
    model_config=dict(n_hidden=MODEL_HIDDEN, n_neurons=MODEL_NEURONS),
    weights=W,
)
with open("pinn_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved: pinn_final.weights.h5, pinn_metadata.json")

In [ ]:
# ============================================================
# CELL 17 — LOSS HISTORY PLOT
# ============================================================

fig, ax = plt.subplots(figsize=(11, 5))
ax.semilogy(history["epoch"], history["total"], lw=1.5, label="total")
ax.semilogy(history["epoch"], history["data"],  lw=1.0, label="data")
ax.semilogy(history["epoch"], history["phys"],  lw=1.0, label="physics")
ax.semilogy(history["epoch"], history["bc"],    lw=1.0, label="boundary")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss (log)")
ax.set_title("PINN Training — 200 snapshots (TensorFlow)")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig("pinn_training_loss.png", dpi=150)
plt.show()

In [ ]:
# ============================================================
# CELL 18 — PREDICTION vs CFD COMPARISON (mid-height slice, last snap)
# ============================================================

last_t = float(times_all[-1])
last_csv_path = snaps[-1][1]

# Load the last snapshot using the same loader from Cell 5
_d, _ = load_snapshot(last_csv_path, col_map=None)
x_cfd, y_cfd, z_cfd = _d["x"], _d["y"], _d["z"]
u_cfd, v_cfd        = _d["u"], _d["v"]

# Take points near mid-height (z ≈ H_CYL/2) within a thin slab
z_target = H_CYL / 2.0
slab = np.abs(z_cfd - z_target) < 3.0

x_s = x_cfd[slab]; y_s = y_cfd[slab]; z_s = z_cfd[slab]
u_s = u_cfd[slab]; v_s = v_cfd[slab]
Vmag_cfd = np.sqrt(u_s**2 + v_s**2)

# PINN prediction at the same points
xn_s = x_s / L_REF; yn_s = y_s / L_REF; zn_s = z_s / L_REF
tn_s = np.full_like(xn_s, last_t / TIME_REF, dtype=np.float32)
xyzt = tf.constant(np.column_stack([xn_s, yn_s, zn_s, tn_s]), dtype=tf.float32)
pred = model(xyzt, training=False).numpy()

u_p = pred[:,0] * V_REF
v_p = pred[:,1] * V_REF
Vmag_p = np.sqrt(u_p**2 + v_p**2)

# Plot (resto igual ao seu código original)
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

def scatter_plot(ax, xvals, yvals, cvals, title, cmap):
    sc = ax.scatter(xvals, yvals, c=cvals, cmap=cmap, s=1)
    theta_p = np.linspace(0, 2*np.pi, 80)
    ax.plot(XC_CYL + R_CYL*np.cos(theta_p), YC_CYL + R_CYL*np.sin(theta_p),
             'k-', lw=1.2)
    ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
    ax.set_title(title)
    ax.set_aspect("equal")
    plt.colorbar(sc, ax=ax, fraction=0.04)

scatter_plot(axes[0,0], x_s, y_s, Vmag_cfd,  f"|V| CFD — t={last_t:.1f}s",   "viridis")
scatter_plot(axes[0,1], x_s, y_s, Vmag_p,    f"|V| PINN — t={last_t:.1f}s",  "viridis")

plt.suptitle(f"CFD vs PINN at z ≈ {z_target:.0f} m, t = {last_t:.1f} s", fontsize=12)
plt.tight_layout()
plt.savefig("pinn_vs_cfd_comparison.png", dpi=150)
plt.show()

# Quick error metrics
err_V = np.abs(Vmag_p - Vmag_cfd)
print(f"\n|V| mean abs error: {err_V.mean():.3f} m/s   max: {err_V.max():.3f} m/s")

In [ ]:
# ============================================================
# CELL 18 — PREDICTION vs CFD COMPARISON (revised)
# ============================================================
# Comparação CFD vs PINN com:
# - Escalas de cor consistentes entre CFD e PINN
# - Suporte para plano XZ (vista lateral) ou XY (vista superior)
# - Múltiplos timesteps configuráveis
# - Erros quantitativos por slice e globais
# - Cilindro desenhado corretamente quando intersecta o slice

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt

# ============================================================
# CONFIGURAÇÃO
# ============================================================
PLANE        = "xz"     # "xz" para vista lateral, "xy" para vista superior
T_SLICES_S   = [10.0, 30.0, 58.0]   # tempos físicos a comparar (segundos)
SLAB_TOL     = 8.0      # half-width do slice [m]
SHOW_FIGS    = True     # exibir as figuras (False = só salvar PNGs)

# Para PLANE="xz", varia y; para PLANE="xy", varia z
if PLANE == "xz":
    SLICE_VALUES = [0, 100, 200, 250, 300, 400, 500, 600, 700]
    SLICE_LABEL  = "y"
else:  # xy
    SLICE_VALUES = [10, 32, 65, 100, 150]  # diferentes alturas
    SLICE_LABEL  = "z"


# ============================================================
# Detecta chaves do dict — robusto a diferentes versões da Cell 5
# ============================================================
_d_test, _ = load_snapshot(snaps[-1][1], col_map=None)
COL_KEYS_AVAILABLE = list(_d_test.keys())
print(f"Chaves disponíveis no snapshot: {COL_KEYS_AVAILABLE}")

# Função para acessar coluna com fallback de nomes
def get_col(d, *candidates):
    """Tenta vários nomes de chave possíveis."""
    for c in candidates:
        if c in d:
            return d[c]
    raise KeyError(f"Nenhuma chave encontrada entre: {candidates}. "
                     f"Disponíveis: {list(d.keys())}")


# ============================================================
# Helpers para desenhar o cilindro em cada plano
# ============================================================
def add_cylinder_xz(ax, y_slice):
    """Cilindro como retângulo lateral (corte XZ). Visível se slice cruza o cilindro."""
    if abs(y_slice - YC_CYL) > R_CYL:
        return
    rect = mpatches.Rectangle(
        (XC_CYL - R_CYL, 0),
        2 * R_CYL, H_CYL,
        linewidth=1.5, edgecolor='black', facecolor='white', zorder=10,
    )
    ax.add_patch(rect)


def add_cylinder_xy(ax, z_slice):
    """Cilindro como círculo (corte XY). Visível se slice está abaixo do topo."""
    if z_slice > H_CYL:
        return
    circle = mpatches.Circle(
        (XC_CYL, YC_CYL), R_CYL,
        linewidth=1.5, edgecolor='black', facecolor='white', zorder=10,
    )
    ax.add_patch(circle)


# ============================================================
# Scatter helper com escala de cor consistente entre CFD e PINN
# ============================================================
def scatter_compare(ax_cfd, ax_pinn, h_coord, v_coord, val_cfd, val_pinn,
                      title_var, unit, cmap, plane, slice_value):
    # Escala única para CFD e PINN: range combinado
    vmin = min(val_cfd.min(), val_pinn.min())
    vmax = max(val_cfd.max(), val_pinn.max())

    sc_cfd = ax_cfd.scatter(h_coord, v_coord, c=val_cfd, cmap=cmap,
                              s=3, vmin=vmin, vmax=vmax, zorder=2)
    sc_pinn = ax_pinn.scatter(h_coord, v_coord, c=val_pinn, cmap=cmap,
                                s=3, vmin=vmin, vmax=vmax, zorder=2)

    # Desenha cilindro nos dois plots
    if plane == "xz":
        add_cylinder_xz(ax_cfd, slice_value)
        add_cylinder_xz(ax_pinn, slice_value)
        xlabel, ylabel = "x [m]", "z [m]"
        xlim = (X_MIN, X_MAX)
        ylim = (Z_MIN, Z_MAX)
    else:  # xy
        add_cylinder_xy(ax_cfd, slice_value)
        add_cylinder_xy(ax_pinn, slice_value)
        xlabel, ylabel = "x [m]", "y [m]"
        xlim = (X_MIN, X_MAX)
        ylim = (Y_MIN, Y_MAX)

    for ax, sc, label in [(ax_cfd, sc_cfd, "CFD"),
                            (ax_pinn, sc_pinn, "PINN")]:
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.set_title(f"{title_var} {label} [{unit}]", fontsize=10)
        ax.set_aspect('equal' if plane == "xy" else 'auto')
        plt.colorbar(sc, ax=ax, fraction=0.04, pad=0.02)


# ============================================================
# Loop principal: para cada tempo, para cada slice
# ============================================================
errors_by_slice = []  # acumula para análise global

for t_target in T_SLICES_S:
    # Encontra o snapshot mais próximo de t_target
    idx_t = np.argmin(np.abs(times_all - t_target))
    actual_t = times_all[idx_t]
    csv_path = snaps[idx_t][1]

    print(f"\n{'='*70}")
    print(f"Tempo: {t_target}s (snapshot mais próximo: t={actual_t:.1f}s)")
    print(f"{'='*70}")

    # Carrega snapshot
    _d, _ = load_snapshot(csv_path, col_map=None)
    x_cfd = get_col(_d, 'x-coordinate', 'x')
    y_cfd = get_col(_d, 'y-coordinate', 'y')
    z_cfd = get_col(_d, 'z-coordinate', 'z')
    u_cfd = get_col(_d, 'x-velocity', 'u')
    v_cfd = get_col(_d, 'y-velocity', 'v')
    w_cfd = get_col(_d, 'z-velocity', 'w')
    P_cfd = get_col(_d, 'dynamic-pressure', 'pressure', 'P')

    # Gauge pressure se necessário
    if P_cfd.mean() > 1e4:
        P_cfd = P_cfd - P_OP

    for slice_value in SLICE_VALUES:
        # Filtro espacial
        if PLANE == "xz":
            slab = np.abs(y_cfd - slice_value) < SLAB_TOL
            h_coord = x_cfd
            v_coord = z_cfd
        else:  # xy
            slab = np.abs(z_cfd - slice_value) < SLAB_TOL
            h_coord = x_cfd
            v_coord = y_cfd

        # Remove interior do cilindro
        inside_cyl = (
            ((x_cfd - XC_CYL)**2 + (y_cfd - YC_CYL)**2 <= R_CYL**2) &
            (z_cfd <= H_CYL)
        )
        valid = slab & ~inside_cyl
        n_valid = valid.sum()

        if n_valid < 10:
            print(f"  [skip] {SLICE_LABEL}={slice_value} — apenas {n_valid} pts")
            continue

        # Arrays do slice
        h_s = h_coord[valid]
        v_s = v_coord[valid]
        x_s = x_cfd[valid]
        y_s = y_cfd[valid]
        z_s = z_cfd[valid]
        u_s = u_cfd[valid]
        v_vel_s = v_cfd[valid]
        w_s = w_cfd[valid]
        P_s = P_cfd[valid]
        Vmag_cfd_s = np.sqrt(u_s**2 + v_vel_s**2 + w_s**2)

        # PINN inference
        xn = (x_s / L_REF).astype(np.float32)
        yn = (y_s / L_REF).astype(np.float32)
        zn = (z_s / L_REF).astype(np.float32)
        tn = np.full_like(xn, actual_t / TIME_REF, dtype=np.float32)
        xyzt = tf.constant(np.column_stack([xn, yn, zn, tn]), dtype=tf.float32)
        pred = model(xyzt, training=False).numpy()
        u_p = pred[:, 0] * V_REF
        v_p = pred[:, 1] * V_REF
        w_p = pred[:, 2] * V_REF
        P_p = pred[:, 4] * P_REF
        Vmag_p_s = np.sqrt(u_p**2 + v_p**2 + w_p**2)

        # Figura 3x2
        fig, axes = plt.subplots(3, 2, figsize=(13, 14))

        scatter_compare(axes[0, 0], axes[0, 1], h_s, v_s,
                          Vmag_cfd_s, Vmag_p_s,
                          "|V|", "m/s", "viridis", PLANE, slice_value)
        scatter_compare(axes[2, 0], axes[2, 1], h_s, v_s,
                          P_s, P_p,
                          "P", "Pa", "coolwarm", PLANE, slice_value)

        plt.suptitle(
            f"CFD vs PINN — plane {PLANE.upper()} at {SLICE_LABEL}={slice_value}m | "
            f"t={actual_t:.1f}s | n={n_valid} pts",
            fontsize=11,
        )
        plt.tight_layout()
        fname = f"pinn_vs_cfd_{PLANE}_{SLICE_LABEL}{slice_value:04d}_t{int(actual_t):03d}s.png"
        plt.savefig(fname, dpi=120, bbox_inches='tight')
        if SHOW_FIGS:
            plt.show()
        plt.close(fig)

        # Errors
        err_V = np.abs(Vmag_p_s - Vmag_cfd_s)
        err_P = np.abs(P_p - P_s)

        errors_by_slice.append({
            't': actual_t,
            'slice_label': SLICE_LABEL,
            'slice_value': slice_value,
            'n_pts': n_valid,
            'V_mae': err_V.mean(), 'V_max': err_V.max(),
            'P_mae': err_P.mean(), 'P_max': err_P.max(),
        })

        print(
            f"  {SLICE_LABEL}={slice_value:4d}m | "
            f"|V| MAE={err_V.mean():5.2f} m/s max={err_V.max():6.2f} | "
            f"P MAE={err_P.mean():5.0f} Pa max={err_P.max():6.0f}"
        )


# ============================================================
# Tabela final consolidada
# ============================================================
print(f"\n{'='*70}")
print("RESUMO GLOBAL DE ERROS")
print(f"{'='*70}")
import pandas as pd
df_errors = pd.DataFrame(errors_by_slice)
print(df_errors.to_string(index=False, float_format='%.3f'))

print(f"\nMédias agregadas:")
print(f"  |V| MAE médio: {df_errors['V_mae'].mean():.3f} m/s")
print(f"  P MAE médio:   {df_errors['P_mae'].mean():.1f} Pa")

In [ ]:
# ============================================================
# CELL 19: ANÁLISE QUANTITATIVA (scatter PINN vs CFD)
# ============================================================
# Plot scatter da predição vs valor real (CFD).
# Foco em V (u, v, w, |V|) e P.
# Usa o ÚLTIMO snapshot completo, todos os pontos.

import matplotlib.pyplot as plt
import numpy as np

# Carrega último snapshot
last_t = float(times_all[-1])
last_csv_path = snaps[-1][1]
_d, _ = load_snapshot(last_csv_path, col_map=None)

x_cfd = _d['x']; y_cfd = _d['y']; z_cfd = _d['z']
u_cfd = _d['u']; v_cfd = _d['v']; w_cfd = _d['w']
P_cfd = _d['P']

if P_cfd.mean() > 1e4:
    P_cfd = P_cfd - P_OP

Vmag_cfd = np.sqrt(u_cfd**2 + v_cfd**2 + w_cfd**2)

# Avalia PINN nos mesmos pontos
n = len(x_cfd)
xyzt = np.column_stack([
    x_cfd / L_REF,
    y_cfd / L_REF,
    z_cfd / L_REF,
    np.full(n, last_t / TIME_REF, dtype=np.float32)
]).astype(np.float32)

pred = model(tf.constant(xyzt), training=False).numpy()
u_p = pred[:, 0] * V_REF
v_p = pred[:, 1] * V_REF
w_p = pred[:, 2] * V_REF
P_p = pred[:, 4] * P_REF
Vmag_p = np.sqrt(u_p**2 + v_p**2 + w_p**2)

# ============================================================
# Plot 1: Scatter PINN vs CFD (5 variáveis: u, v, w, |V|, P)
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

datasets = [
    (u_cfd, u_p, 'u', 'm/s', axes[0, 0]),
    (v_cfd, v_p, 'v', 'm/s', axes[0, 1]),
    (w_cfd, w_p, 'w', 'm/s', axes[0, 2]),
    (Vmag_cfd, Vmag_p, '|V|', 'm/s', axes[1, 0]),
    (P_cfd, P_p, 'P', 'Pa', axes[1, 1]),
]

for cfd_vals, pinn_vals, name, unit, ax in datasets:
    # Scatter (com transparência por densidade)
    ax.scatter(cfd_vals, pinn_vals, s=1, alpha=0.3, c='steelblue')

    # Linha y=x (predição perfeita)
    vmin = min(cfd_vals.min(), pinn_vals.min())
    vmax = max(cfd_vals.max(), pinn_vals.max())
    ax.plot([vmin, vmax], [vmin, vmax], 'r--', lw=1.5, label='y=x (ideal)')

    # Métricas
    mae = np.mean(np.abs(pinn_vals - cfd_vals))
    rmse = np.sqrt(np.mean((pinn_vals - cfd_vals)**2))
    bias = np.mean(pinn_vals - cfd_vals)

    # Correlação
    corr = np.corrcoef(cfd_vals, pinn_vals)[0, 1]

    ax.set_xlabel(f'{name} CFD [{unit}]')
    ax.set_ylabel(f'{name} PINN [{unit}]')
    ax.set_title(f'{name}: MAE={mae:.2f} | RMSE={rmse:.2f} | bias={bias:+.2f} | r={corr:.3f}')
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_aspect('equal')

# Remove eixo vazio
axes[1, 2].axis('off')

plt.suptitle(f'PINN vs CFD scatter — t={last_t:.1f}s | n={n} pts', fontsize=12)
plt.tight_layout()
plt.savefig('pinn_vs_cfd_scatter.png', dpi=120, bbox_inches='tight')
plt.show()


# ============================================================
# Plot 2: Histograma de erros
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

datasets = [
    (u_p - u_cfd, 'u', 'm/s', axes[0, 0]),
    (v_p - v_cfd, 'v', 'm/s', axes[0, 1]),
    (w_p - w_cfd, 'w', 'm/s', axes[0, 2]),
    (Vmag_p - Vmag_cfd, '|V|', 'm/s', axes[1, 0]),
    (P_p - P_cfd, 'P', 'Pa', axes[1, 1]),
]

for errors, name, unit, ax in datasets:
    ax.hist(errors, bins=100, alpha=0.7, edgecolor='black', linewidth=0.5)
    ax.axvline(0, color='red', ls='--', lw=1.5, label='zero error')
    ax.axvline(np.median(errors), color='green', ls=':', lw=1.5,
                label=f'median={np.median(errors):.2f}')

    # Estatísticas
    pct25 = np.percentile(errors, 25)
    pct75 = np.percentile(errors, 75)

    ax.set_xlabel(f'Erro {name} (PINN - CFD) [{unit}]')
    ax.set_ylabel('Frequência')
    ax.set_title(f'{name}: Q1={pct25:.2f}, Q3={pct75:.2f}, IQR={pct75-pct25:.2f}')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

axes[1, 2].axis('off')

plt.suptitle(f'Distribuição de erros PINN - CFD — t={last_t:.1f}s', fontsize=12)
plt.tight_layout()
plt.savefig('pinn_vs_cfd_errors_hist.png', dpi=120, bbox_inches='tight')
plt.show()


# ============================================================
# Tabela resumo
# ============================================================
print("\n" + "="*70)
print(f"RESUMO PINN vs CFD em t={last_t:.1f}s (n={n} pts)")
print("="*70)
print(f"\n{'Var':<8} {'MAE':>10} {'RMSE':>10} {'Bias':>10} {'Corr':>8} {'Relativa MAE':>14}")
print("-" * 70)

scales = {'u': V_REF, 'v': V_REF, 'w': V_REF, '|V|': V_REF, 'P': 100}

for cfd_vals, pinn_vals, name, unit, _ in datasets:
    if isinstance(cfd_vals, tuple):
        continue
    mae = np.mean(np.abs(pinn_vals - cfd_vals))
    rmse = np.sqrt(np.mean((pinn_vals - cfd_vals)**2))
    bias = np.mean(pinn_vals - cfd_vals)
    corr = np.corrcoef(cfd_vals, pinn_vals)[0, 1]
    scale = scales.get(name, 1.0)
    rel_mae = 100 * mae / scale
    print(f"{name:<8} {mae:>9.3f}  {rmse:>9.3f}  {bias:>+9.3f}  {corr:>7.3f}  {rel_mae:>12.1f}%")